# PointNet++ Training experimental

This notebook serves the purpose of creating the training routine for the PointNet++ to encode point clouds into the same latent space as the CAD sequence encoding done by the DeepCAD model.

The pointnet venv is located at "Point-Cloud-Reconstruction/pointnet.pytorch", but the model used is in "Point-Cloud-Reconstruction/Pointnet_Pointnet2_pytorch"

## 1. Check point cloud data
We will use open3d to inspect the point cloud files. During the conversion of the CAD sequences to point clouds errors appeared, which is why I assume there must be some corrupt point cloud files, which need to be excluded.

- Load train/val/test split
- create train/val/test pc path lists

In [126]:
import open3d as o3d
import os
from tqdm import tqdm
from glob import glob
import h5py
import numpy as np

In [167]:
DATA_ROOT = "../data"

PC_ROOT = os.path.join(DATA_ROOT, "pc_cad")
SPLIT = os.path.join(DATA_ROOT, "train_val_test_split.json")

Here the split file is opened, it contains the information about all files in the dataset

In [128]:
with open(SPLIT, "r") as fp:
    all_data = json.load(fp)
print(f"Number of samples that should be in the split: {len(all_data['train']) + len(all_data['validation']) + len(all_data['test'])}")
for phase in all_data.keys():
    print(phase, len(all_data[phase]))

Number of samples that should be in the split: 178238
train 161240
validation 8946
test 8052


In [129]:
train = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['train']]
val = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['validation']]
test = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['test']]

Now we need to check if the point cloud files in the split are actually there and not corrupt. The point cloud files are generated using json2pc.py, which can fail from time to time.

In [130]:
file_pattern = "**/*.ply"
all_files = glob(f"{PC_ROOT}/{file_pattern}", recursive=True)
print(f"Total number of files: {len(all_files)}")
print(f"Missing files: {len(train)+len(val)+len(test)-len(all_files)}")

Total number of files: 177948
Missing files: 290


Apparently in 290 cases no point cloud file was created. These have to be removed from the split. Therefore we filter all files in the split by the files which are actually on disk. Then we obtain the indices of the removed files, in order to later remove the respective latent representations.

In [150]:
all_files_set = set(all_files)

train_pc = [entry for entry in train if entry in all_files_set]
corrupt_files_idx_train = [i for i, entry in enumerate(train) if entry not in all_files_set]

val_pc = [entry for entry in val if entry in all_files_set]
corrupt_files_idx_val = [i for i, entry in enumerate(val) if entry not in all_files_set]

test_pc = [entry for entry in test if entry in all_files_set]
corrupt_files_idx_test = [i for i, entry in enumerate(test) if entry not in all_files_set]

In [151]:
assert (len(train_pc)+len(val_pc)+len(test_pc) == len(all_files))

In [133]:
def check_valid_pc(paths_list):
    corrupt_counter = 0
    for pc_file in tqdm(paths_list):
        try:
            pc = o3d.io.read_point_cloud(pc_file)
            if not pc.has_points():
                corrupt_counter += 1
        except Exception as e:
            corrupt_counter += 1
    print(f"There are {corrupt_counter} corrupt files")
    if corrupt_counter == 0:
        return True
    else:
        return False

In [134]:
print("Check train set")
assert(check_valid_pc(train_pc))
print("Check val set")
assert(check_valid_pc(val_pc))
print("Check test set")
assert(check_valid_pc(test_pc))

Check train set


100%|█████████████████████████████████| 160982/160982 [01:54<00:00, 1411.71it/s]


There are 0 corrupt files
Check val set


100%|█████████████████████████████████████| 8928/8928 [00:06<00:00, 1390.14it/s]


There are 0 corrupt files
Check test set


100%|█████████████████████████████████████| 8038/8038 [00:05<00:00, 1369.33it/s]

There are 0 corrupt files


## Check latent representation

In [135]:
LATENT_ROOT = os.path.join(DATA_ROOT, "latent/pretrained/results/all_zs_ckpt1000.h5")

In [136]:
with h5py.File(LATENT_ROOT, 'r') as f:
    for key in f.keys():
        print(f"key - {key}: {f[key]}")
    train_latent = np.array(f["train_zs"])
    val_latent = np.array(f["validation_zs"])
    test_latent = np.array(f["test_zs"])

key - test_zs: <HDF5 dataset "test_zs": shape (8052, 256), type "<f4">
key - train_zs: <HDF5 dataset "train_zs": shape (161240, 256), type "<f4">
key - validation_zs: <HDF5 dataset "validation_zs": shape (8946, 256), type "<f4">


In [137]:
def check_latent_valid(data):
    # Zero rows
    zero_rows = np.all(embeddings == 0, axis=1)
    num_zero_rows = np.sum(zero_rows)
    if num_zero_rows > 0:
        print(f"There are {num_zero_rows} zero rows.")
    
    # NaN or Inf values
    nan_or_inf_rows = np.any(np.isnan(embeddings) | np.isinf(embeddings), axis=1)
    num_nan_or_inf_rows = np.sum(nan_or_inf_rows)
    if num_nan_or_inf_rows > 0:
        print(f"There are {num_nan_or_inf_rows} rows with NaN or Inf values.")
    
    if num_zero_rows == 0 and num_nan_or_inf_rows == 0:
        print("All latent represenations are valid!\n")
        return True
    else:
        return False

print("Check latent train set")
assert(check_latent_valid(train_latent))
print("Check latent val set")
assert(check_latent_valid(val_latent))
print("Check latent test set")
assert(check_latent_valid(test_latent))

Check latent train set
All latent represenations are valid!

Check latent val set
All latent represenations are valid!

Check latent test set
All latent represenations are valid!



## Allign point clouds and latent representations

We removed some point clouds due to conversion errors from json. However the latent representation is available for every sample. Now we need to allign the point clouds with the latent representations, i.e. we need to remove the latent representations of the corrupt point clouds. First we have to find out the indices of the removed point clouds in each set.

Now here I assume that the order of latent representations is the same as the order of point clouds. Looking at the encoding script in `test.py`, the dataloaders are initiated without shuffle, which backs up my assumption.

In [154]:
train_latent = np.delete(train_latent, corrupt_files_idx_train, axis = 0)
val_latent = np.delete(val_latent, corrupt_files_idx_val, axis = 0)
test_latent = np.delete(test_latent, corrupt_files_idx_test, axis = 0)

# Training

After we scrutinized and checked the data, let's start building the training routine. This will be the `main()` method for training. The important thing to do here is to implement a dataset class.

In [172]:
from torch.utils.data import Dataset

class PointCloudToEncoding(Dataset):
    
    def __init__(self, root, split):
        self.root = root
        self.split = split
        self.split_path = os.path.join(root, "train_val_test_split.json")
        self.pc_path = os.path.join(root, "pc_cad")
        self.latent_path = os.path.join(root, "latent/pretrained/results/all_zs_ckpt1000.h5")
        
    def __len__(self):
        pass
        
    def __getitem__(self, idx):
        pass

    def read_split(self):
        with open(self.split_path, "r") as fp:
            all_data = json.load(fp)
        print(f"Number of samples that should be in the {self.split} set: {len(all_data[self.split])}")

In [175]:
train_dataset = PointCloudToEncoding(DATA_ROOT, 'train')

In [176]:
train_dataset.read_split()

Number of samples that should be in the train set: 161240


In [113]:
problematic_file = "/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/cad_json/0011/00116212.json"
try:
    with open(problematic_file, "r") as f:
        data = json.load(f)
    print("File loaded successfully.")
    print(json.dumps(data, indent=4))  # Pretty-print the JSON content.
except json.JSONDecodeError as e:
    print(f"JSON decoding error: {e}")
except Exception as e:
    print(f"Error reading file: {e}")

File loaded successfully.
{
    "entities": {
        "FRAcwEqNExbvyOz_2": {
            "name": "Extrude 3",
            "type": "ExtrudeFeature",
            "profiles": [
                {
                    "profile": "JNC",
                    "sketch": "FRSAyqbsz52iKa9_2"
                }
            ],
            "extent_two": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AgainstDistance",
                    "name": "none",
                    "value": 0.0
                },
                "type": "DistanceExtentDefinition",
                "taper_angle": {
                    "type": "ModelParameter",
                    "role": "Side2TaperAngle",
                    "name": "none",
                    "value": 0.0
                }
            },
            "extent_one": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AlongDistance",
                   

Test: 14 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Val: 18 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Train: Irgendwo findet ein Parallels problem statt (0, 78950, 1, 82290)Es ist die file 0011/00116212

Error kommt von process_one -> create_CAD (in visualize.py)

Laut stackoverflow soll man mit ulimit -s stack anschauen und mit ulimit -s \<neuerWert> erhöhen

Chat GPT: "On macOS (and other Unix-like operating systems), the ulimit -s command is used to query or set the stack size limit for processes. The stack size determines the amount of memory allocated for a program's stack, which is used for function calls, local variables, and control flow."

